# Databricks RAGアプリチュートリアル - 2.（別解）Agent Bricks Knowledge Assistant によるRAG構築

このノートブックは、`2_RAGエージェントの構築.ipynb`（LangGraph でエージェントを自作する方式）の**オルタナティブ版**です。同じ「PDFドキュメントに対するRAG（Retrieval-Augmented Generation）」を、**Agent Bricks の Knowledge Assistant** を使ってノーコード／ローコードで構築します。

## 2つの方式の違い

| 観点 | 既存ノートブック（Agent Bricks Custom Agents） | このノートブック（Knowledge Assistant） |
|---|---|---|
| 実装方法 | `agent.py` を自作（LangGraph でツール呼び出しエージェントを構築） | UI もしくは SDK でエージェントを宣言的に作成 |
| RAGパイプライン | AI Search Index・retriever・チャンキングを**自前で設計** | パース・チャンク化・埋め込み・検索・引用生成が**フルマネージド** |
| デプロイ | `log_model` → `register_model` → `agents.deploy()` を自分で実行 | エンドポイントが**自動生成**される |
| 引用（citations） | 自前で実装が必要 | **標準で付与**される |
| 品質改善 | プロンプト・コードを手で調整 | 専門家の自然言語フィードバック（ガイドライン）で改善 |
| 向いている用途 | 複雑なツール呼び出し・マルチステップ制御 | **ドキュメントQ&A** に特化 |

> 💡 **使い分け**: 「社内ドキュメントへのQ&Aチャットボット」を素早く高品質に作りたいなら Knowledge Assistant、独自ツール連携や複雑なワークフロー制御が必要なら Agent Bricks Custom Agents（既存ノートブック）が適しています。

## このノートブックで学習する内容

1. **Agent Bricks / Knowledge Assistant** の概要と要件
2. **UI での作成手順**（画面操作ベース）
3. **SDK / REST API での作成手順**（自動化・再現性重視）
4. 作成したエージェント**エンドポイントへのクエリ**
5. **MLflow による評価**との連携（概要）

## 実行環境

このノートブックは**サーバーレスコンピュート**での実行を想定しています。

## 参考リンク

- [Knowledge Assistant ドキュメント](https://docs.databricks.com/aws/en/agents/agent-bricks/knowledge-assistant)
- [Agent Bricks 製品ページ](https://www.databricks.com/product/artificial-intelligence/agent-bricks)
- [Knowledge Assistant REST API リファレンス](https://docs.databricks.com/api/workspace/knowledgeassistants)


## 1. Agent Bricks と Knowledge Assistant とは

### Agent Bricks

**Agent Bricks** は、エンタープライズ向けの AI エージェントを構築・デプロイ・ガバナンスするための Databricks の統合プラットフォームです。データからモデルまでを単一のガバナンス基盤（Unity Catalog）上で管理できます。主なエージェントタイプに、**Knowledge Assistant**、**Supervisor Agent**、**Custom LLM** などがあります。

### Knowledge Assistant

**Knowledge Assistant** は、ドキュメントに対する **Q&Aチャットボット**を構築するためのエージェントタイプで、**2026年1月に一般提供（GA）** されました。

- **Instructed Retriever** という手法により、従来のRAGの限界に対処し、**引用（citations）付き**の高品質な回答を返します。
- ドキュメントの**パース・チャンク化・埋め込み・検索・引用生成をすべて自動**で行います（`ai_parse_document` を内部で使用）。
- **多言語対応**。製品ドキュメントQ&A、社内規定の照会、カスタマーサポートのナレッジベースなどに向いています。

> ⚠️ **REST API のステータス**: Knowledge Assistant 機能自体は GA ですが、後述の REST API 操作は **Beta** 表記です（将来パスやフィールドが変わる可能性があります）。


## 2. 前提条件・要件

Knowledge Assistant を使うには、ワークスペースが以下を満たす必要があります。

- **サーバーレスコンピュート**が有効（Unity Catalog 有効ワークスペースではデフォルトで有効）
- **Unity Catalog** が有効
- **Databricks Model Serving** へのアクセス権
- 予算が非ゼロの **serverless usage policy**
- **サポート対象リージョン**であること

> 🗾 **東京リージョンの注意**: Knowledge Assistant は一部リージョンでは **cross-geography routing の有効化が必要**です。東京（`ap-northeast-1`）はこれに該当するため、利用時はワークスペース管理者に確認してください。最新のリージョン対応は [Feature region support](https://docs.databricks.com/aws/en/resources/feature-region-support) を参照。

### ナレッジソースの制約

| 項目 | 制約 |
|---|---|
| 1エージェントあたりのソース数 | 最大 **10** |
| ファイルサイズ | **50MB 超**はスキップ |
| ページ数 | PDF/DOC/DOCX/PPT/PPTX で **500ページ超**はスキップ（TXT/MD は制限なし） |
| ファイル名 | `_` または `.` で始まるファイルはスキップ |
| 対応ファイル形式（Volume） | txt, pdf, md, ppt/pptx, doc/docx |

### このノートブックの前提

`1_PDFのパースとベクトルインデックスの作成.ipynb` で、UC Volume に PDF を配置済みであることを前提とします（例: `/Volumes/{catalog}/{schema}/{volume}/`）。そのPDFをナレッジソースとして Knowledge Assistant を作成します。


## 3. ライブラリの準備

Knowledge Assistant の作成・管理には **Databricks SDK for Python**（`databricks-sdk`）を使用します。サーバーレス環境には標準で含まれていますが、最新機能（`knowledge_assistants` サービス）を使うため更新します。

In [ ]:
# Databricks SDK を最新化（knowledge_assistants サービスを利用するため）
%pip install -U -qqqq databricks-sdk mlflow
dbutils.library.restartPython()

## 4. パラメータ設定（widget）

既存ノートブックと同様に、環境依存の値は **widget** で指定します。上部の入力欄で自分の環境に合わせて変更し、このセルを実行してください。

> ⚠️ **`KA_DISPLAY_NAME` の制約**: Knowledge Assistant の表示名は、正規表現 `^[\w.-]+$`（英数字・アンダースコア・ドット・ハイフンのみ）かつ **4〜63文字**という制約があります。**日本語や空白は使用できません**。ワークスペース内で一意である必要があります。

In [ ]:
# パラメータ設定（widget から取得）
dbutils.widgets.text("CATALOG_NAME", "skato", "カタログ名")
dbutils.widgets.text("SCHEMA_NAME", "rag_workshop", "スキーマ名")
dbutils.widgets.text("VOLUME_NAME", "pdf_files", "ボリューム名（PDF配置先）")
# 表示名は ^[\w.-]+$ / 4〜63文字 / 日本語・空白不可
dbutils.widgets.text("KA_DISPLAY_NAME", "genai-doc-assistant", "Knowledge Assistant 表示名")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VOLUME_NAME = dbutils.widgets.get("VOLUME_NAME")
KA_DISPLAY_NAME = dbutils.widgets.get("KA_DISPLAY_NAME")

# ナレッジソースとして使う Volume パス
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

print(f"カタログ: {CATALOG_NAME}")
print(f"スキーマ: {SCHEMA_NAME}")
print(f"ボリュームパス: {VOLUME_PATH}")
print(f"Knowledge Assistant 表示名: {KA_DISPLAY_NAME}")

# 表示名の簡易バリデーション（作成前に気づけるように）
import re
assert re.fullmatch(r"[\w.-]{4,63}", KA_DISPLAY_NAME), \
    "KA_DISPLAY_NAME は英数字・_・.・- のみ、4〜63文字にしてください（日本語・空白不可）"

## 5. 【方式A】UI での作成手順

まずはコードを書かずに、Databricks の画面上で Knowledge Assistant を作成する手順です。**実際の操作はこのノートブックの外（左ナビ）で行います**。手順を追いながら操作してください。

### エージェント作成への入り方

1. ワークスペース左ナビの **Agents**（ユーザー＋スパークルのアイコン）をクリック
2. **Create Agent** ボタンをクリック
3. 表示された選択肢から **Knowledge Assistant** を選択

### Step 1: Configure your agent（設定）

1. **Name** … エージェント名を入力
2. **Description** … このエージェントができることを記述（例: 「生成AI開発に関する技術文書へのQ&A」）
3. **Knowledge source** パネルでナレッジソースを追加（**最大10ソース**）。ダイアログの **Type** ドロップダウンで種類を選びます:
   - **Files in a Volume**（今回はこれ）: **Source** で PDF を含む UC Volume を選択 → **Name** と **Describe the content**（内容の説明）を入力
   - **Files in a Table**: `content` 列とメタデータ列を持つ UC テーブルを指定
   - **AI Search Index**: AI Search インデックスを指定（**Doc URI Column** と **Text Column** を選択）
4. （任意）**Instructions** … 回答方法のガイドラインを入力（例: 「必ず日本語で、引用付きで回答する」）
5. **Create Agent** をクリック

> ⏳ 作成とナレッジソースの同期には**最大で数時間**かかることがあります。準備完了後、右サイドパネルに同期済みソースが表示されます。

### Step 2: Test your agent（テスト）

- **Build** タブでエージェントと直接チャット、または **Open in Playground** で Agent Bricks AI Playground でテスト
- 回答の評価に使えるビュー:
  - **View thoughts** … 回答に至る思考過程
  - **View trace** … フルトレース（UI でラベル付けして品質を追跡可能）
  - **View sources** … 引用したファイル一覧

### Step 3: Improve quality（品質改善）

- **Examples** タブで想定質問や誤答した質問を追加（**+ Add** → 質問入力 → **Add**）
- 質問をクリックし **Guidelines**（回答方針）を追加（保存直後に反映）
- 専門家に共有してフィードバックを収集（**CAN_MANAGE** 権限が必要）

### 作成後の確認

- エージェントページの **Endpoint** で、自動生成された Databricks Model Serving エンドポイントの詳細を確認
- **Open in playground → Get code** で、`Curl API` / `Python API` の呼び出しコードを取得できます


## 6. 【方式B】SDK / REST API での作成（自動化・再現性重視）

ここからは **Databricks SDK for Python** を使い、コードで Knowledge Assistant を作成します。ハンズオンの再現性を高めたい場合や、CI/CD に組み込みたい場合はこちらが便利です。

内部的には以下の REST API を呼び出します（いずれも **Beta**）:

| 操作 | メソッド + パス |
|---|---|
| KA 作成 | `POST /api/2.1/knowledge-assistants` |
| ナレッジソース追加 | `POST /api/2.1/knowledge-assistants/{id}/knowledge-sources` |
| ナレッジソース同期 | `POST /api/2.1/knowledge-assistants/{id}/knowledge-sources:sync` |

> 📖 API 仕様: [Create a Knowledge Assistant](https://docs.databricks.com/api/workspace/knowledgeassistants/createknowledgeassistant)

### 6-1. WorkspaceClient の初期化

In [ ]:
from databricks.sdk import WorkspaceClient

# ノートブック実行ユーザーの認証情報が自動的に使われます（PAT 不要）
w = WorkspaceClient()

print("WorkspaceClient を初期化しました")
print(f"接続先ホスト: {w.config.host}")

### 6-2. Knowledge Assistant の作成

`create_knowledge_assistant` で本体を作成します。作成時に必須なのは `display_name` と `description` です。`instructions` は任意で、回答生成時のグローバルな指示になります。

作成レスポンスには、**推論クエリに使うエンドポイント名（`endpoint_name`）** や、リソース名（`name`、`knowledge-assistants/{id}` 形式）が含まれます。この `name` を後続のソース追加・同期で使います。

In [ ]:
from databricks.sdk.service.knowledgeassistants import KnowledgeAssistant

ka = w.knowledge_assistants.create_knowledge_assistant(
    knowledge_assistant=KnowledgeAssistant(
        display_name=KA_DISPLAY_NAME,
        description="生成AI開発に関する技術文書（GenAI開発ワークフロー、MLOps、エージェント設計など）へのQ&Aアシスタント",
        instructions=(
            "あなたは生成AI開発に関する専門アシスタントです。"
            "検索された文書の内容に基づき、日本語で丁寧に、引用を添えて回答してください。"
            "不確かな情報は推測せず、分からない場合はその旨を伝えてください。"
        ),
    )
)

# リソース名（knowledge-assistants/{id}）とエンドポイント名を控える
KA_NAME = ka.name                 # 例: "knowledge-assistants/xxxxxxxx"
KA_ENDPOINT_NAME = ka.endpoint_name

print("=== Knowledge Assistant を作成しました ===")
print(f"リソース名 (name): {KA_NAME}")
print(f"エンドポイント名 (endpoint_name): {KA_ENDPOINT_NAME}")
print(f"状態 (state): {ka.state}")   # 作成直後は CREATING

### 6-3. ナレッジソースの追加（Volume の PDF）

作成した Knowledge Assistant に、`1_...` ノートブックで PDF を配置した UC Volume をナレッジソースとして追加します。

- `parent` には手順 6-2 の `KA_NAME`（`knowledge-assistants/{id}`）を渡します
- `source_type="files"` を指定し、`FilesSpec(path=...)` に Volume パスを渡します
- AI Search インデックスを使いたい場合は、代わりに `source_type="index"` と `IndexSpec(...)` を使います（下のセルのコメント参照）

> ⚠️ **AI Search Index を使う場合の注意**: 埋め込みモデルが `databricks-gte-large-en` / `databricks-bge-large-en` / `databricks-qwen3-embedding-0-6b` のいずれかである必要があります。`1_...` で日本語埋め込みモデル（plamo）を使ったインデックスは非対応の可能性があるため、ここでは **Volume のファイルを直接ソースにする**方式を採用しています。

In [ ]:
from databricks.sdk.service.knowledgeassistants import KnowledgeSource, FilesSpec

source = w.knowledge_assistants.create_knowledge_source(
    parent=KA_NAME,
    knowledge_source=KnowledgeSource(
        display_name="genai-pdf-docs",
        description="生成AI開発に関するPDF技術文書（エージェント設計パターン、GenAI開発ワークフロー等）",
        source_type="files",
        files=FilesSpec(path=VOLUME_PATH),
    ),
)

KS_NAME = source.name  # 例: "knowledge-assistants/{id}/knowledge-sources/{ks_id}"
print("=== ナレッジソースを追加しました ===")
print(f"ソース名: {KS_NAME}")
print(f"状態: {source.state}")

# --- 参考: AI Search Index をソースにする場合 ---
# from databricks.sdk.service.knowledgeassistants import IndexSpec
# source = w.knowledge_assistants.create_knowledge_source(
#     parent=KA_NAME,
#     knowledge_source=KnowledgeSource(
#         display_name="genai-vs-index",
#         description="生成AI文書のAI Searchインデックス",
#         source_type="index",
#         index=IndexSpec(
#             index_name=f"{CATALOG_NAME}.{SCHEMA_NAME}.chunked_document_vs_index",
#             text_col="chunk_content",
#             doc_uri_col="path",
#         ),
#     ),
# )

### 6-4. ナレッジソースの同期（Sync）

Volume やテーブルのファイルソースは、追加・更新後に **Sync（同期）** が必要です（インデックスソースは自動更新のため不要）。同期により、ドキュメントのパース・チャンク化・インデックス作成が実行されます。

> ⏳ 同期は**数時間かかる**ことがあります。下の状態確認セルで `ACTIVE` になるのを待ってから、クエリに進んでください。

In [ ]:
# ファイルソースを同期（パース・インデックス作成を開始）
w.knowledge_assistants.sync_knowledge_sources(name=KA_NAME)
print(f"同期を開始しました: {KA_NAME}")
print("※ 完了まで時間がかかります。次のセルで状態を確認してください。")

### 6-5. 状態の確認

`get_knowledge_assistant` でエージェントの状態を確認します。`state` が **`ACTIVE`** になれば準備完了です（`CREATING` は準備中、`FAILED` は失敗）。

In [ ]:
current = w.knowledge_assistants.get_knowledge_assistant(name=KA_NAME)
print(f"Knowledge Assistant: {current.display_name}")
print(f"状態 (state): {current.state}")   # CREATING / ACTIVE / FAILED
print(f"エンドポイント名: {current.endpoint_name}")
if current.error_info:
    print(f"エラー情報: {current.error_info}")

# ナレッジソースの状態も確認
sources = w.knowledge_assistants.list_knowledge_sources(parent=KA_NAME)
for s in sources:
    print(f"  - ソース '{s.display_name}': state={s.state}")

## 7. エンドポイントへのクエリ（質問を投げる）

Knowledge Assistant は、作成時に **Databricks Model Serving エンドポイント**を自動生成します。エンドポイント名は手順 6-2 で取得した **`endpoint_name`** です（命名規則を仮定せず、この値を使うのが確実です）。

クエリは標準の Databricks Model Serving 呼び出しで、**OpenAI 互換の `messages` 形式**です。

> ⚠️ 状態が `ACTIVE` になってから実行してください。準備中の場合はエラーになります。

In [ ]:
# SDK 経由でエンドポイントにクエリ（OpenAI 互換 messages 形式）
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

question = "生成AIの開発ワークフローについて、重要なステップを教えてください"

response = w.serving_endpoints.query(
    name=KA_ENDPOINT_NAME,
    messages=[
        ChatMessage(role=ChatMessageRole.USER, content=question),
    ],
)

print(f"質問: {question}\n")
print("=== 回答 ===")
print(response.choices[0].message.content)

### 参考: REST API を直接呼ぶ場合（curl 相当）

SDK を使わずに直接エンドポイントを叩く場合は、以下のように `POST /serving-endpoints/{endpoint_name}/invocations` を呼びます。UI の **Open in playground → Get code** でも、実際の呼び出しコードを取得できます。

```bash
# 事前に DATABRICKS_HOST, DATABRICKS_TOKEN を設定
curl -X POST "$DATABRICKS_HOST/serving-endpoints/<endpoint_name>/invocations" \
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \
  -H "Content-Type: application/json" \
  -d '{ "messages": [ { "role": "user", "content": "質問文" } ] }'
```


## 8. MLflow による評価との連携

Knowledge Assistant は MLflow とネイティブに統合されており、UI 上でトレース（**View trace**）を確認し、ラベルを付けて品質を追跡できます。加えて、MLflow 3 の GenAI 評価機能（`mlflow.genai.evaluate`）を使って、エンドポイントをプログラム的に評価できます。

下の例は、いくつかの質問に対する回答を組み込みスコアラー（関連性・安全性）で評価する最小構成です。より本格的な評価（合成データ生成・カスタムスコアラー・LLM-as-a-Judge・プロンプトレジストリ）は、姉妹ノートブック **MLflow評価詳細版** で扱います。

In [ ]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, Safety

# エンドポイントを叩く predict 関数（評価対象）
def predict_fn(messages):
    resp = w.serving_endpoints.query(name=KA_ENDPOINT_NAME, messages=messages)
    return resp.choices[0].message.content

# 評価用データセット（必要に応じて拡張してください）
eval_dataset = [
    {"inputs": {"messages": [{"role": "user", "content": "生成AIの開発ワークフローについて教えてください"}]}},
    {"inputs": {"messages": [{"role": "user", "content": "エージェントシステムの設計パターンにはどんなものがありますか？"}]}},
    {"inputs": {"messages": [{"role": "user", "content": "MLOpsのベストプラクティスは何ですか？"}]}},
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda messages: predict_fn(messages),
    scorers=[
        RelevanceToQuery(),  # 質問に対する回答の関連性
        Safety(),            # 安全性（有害性の検出）
    ],
)

print("評価が完了しました。MLflow UI で詳細な結果を確認してください。")

## 9. クリーンアップ（任意）

ハンズオンで作成した Knowledge Assistant を削除する場合は、以下を実行します。**削除すると自動生成されたエンドポイントも削除**されます。不要な場合はこのセルを実行しないでください。

In [ ]:
# 作成した Knowledge Assistant を削除（自動生成エンドポイントも削除される）
# 実行する場合は下の行のコメントを外してください
# w.knowledge_assistants.delete_knowledge_assistant(name=KA_NAME)
# print(f"削除しました: {KA_NAME}")
print("クリーンアップはコメントアウトされています。必要に応じて有効化してください。")

## 10. まとめと次のステップ

### このノートブックで学んだこと

- **Agent Bricks / Knowledge Assistant** の概要と、Agent Bricks Custom Agents 方式との違い
- **UI での作成手順**（Agents → Create Agent → Knowledge Assistant の3ステップ）
- **SDK / REST API での作成**（`create_knowledge_assistant` → `create_knowledge_source` → `sync_knowledge_sources`）
- 自動生成された**エンドポイントへのクエリ**（`endpoint_name` を使った OpenAI 互換呼び出し）
- **MLflow 評価**との連携の入り口

### 次のステップ

1. **品質改善**: UI の Examples タブでガイドラインを追加し、回答品質を高める
2. **詳細な評価**: 姉妹ノートブック（MLflow 評価詳細版）で、合成データ生成・カスタムスコアラー・LLM-as-a-Judge・プロンプトレジストリを学ぶ
3. **Web アプリ化**: `3_Webアプリケーションのデプロイ.ipynb` の手順で、このエンドポイントを Databricks Apps から利用する（`SERVING_ENDPOINT` に `endpoint_name` を指定）

### 参考リンク

- [Knowledge Assistant ドキュメント](https://docs.databricks.com/aws/en/agents/agent-bricks/knowledge-assistant)
- [Knowledge Assistant REST API リファレンス](https://docs.databricks.com/api/workspace/knowledgeassistants)
- [Agent Bricks 製品ページ](https://www.databricks.com/product/artificial-intelligence/agent-bricks)
- [MLflow GenAI 評価](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/)
